# Lecture 2 — Class Exercise
## Bar Charts: World Happiness Report 2023

Two polished bar charts (plus the stretch goal) following the four rules: zero baseline, insight titles, clean professional styling, horizontal bars for long category names.

In [ ]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

In [ ]:
# Shared style so both charts look like one family
HIGHLIGHT = '#0F766E'   # teal: the story
MUTED     = '#C5CCD3'   # light grey: context
WARM      = '#E2674A'   # coral: contrast group
INK       = '#1F2937'
GRID      = '#E5E7EB'
FONT      = dict(family='Inter, Helvetica, Arial, sans-serif', size=14, color=INK)

def style(fig, height=520):
    fig.update_layout(
        template='plotly_white',
        font=FONT,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(l=20, r=40, t=100, b=60),
        showlegend=False,
        title=dict(x=0.0, xanchor='left', font=dict(size=20)),
    )
    fig.update_xaxes(showgrid=True, gridcolor=GRID, zeroline=True,
                     zerolinecolor=INK, zerolinewidth=1.5)
    fig.update_yaxes(showgrid=False, automargin=True)
    return fig

## Task 1 — Regional Comparison Bar Chart

In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending: Plotly draws the first bar at the bottom

print(region_avg)

# Step 2: Build your chart
global_avg = df['Happiness_Score'].mean()
top_region = region_avg.iloc[-1]
low_region = region_avg.iloc[0]
gap = top_region['Happiness_Score'] - low_region['Happiness_Score']

# Design choice beyond default: highlight the standout region, mute the rest
colors = [HIGHLIGHT if r == top_region['Region'] else MUTED for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.2f}' for v in region_avg['Happiness_Score']],
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}<br>Avg score: %{x:.2f}<extra></extra>',
))

fig1.add_vline(x=global_avg, line_dash='dot', line_color='#475569',
               annotation_text=f'Global avg {global_avg:.2f}',
               annotation_position='bottom right',
               annotation_font=dict(size=12, color='#475569'))

fig1.update_layout(
    title=dict(text=(f"<b>{top_region['Region']} is the happiest region, "
                     f"{gap:.1f} points ahead of {low_region['Region']}</b>"
                     "<br><sup>Average happiness score by region (0–10 scale), "
                     "World Happiness Report 2023</sup>")),
)
fig1.update_xaxes(range=[0, region_avg['Happiness_Score'].max() * 1.15],  # zero baseline
                  title='Average happiness score')
style(fig1, height=max(420, 55 * len(region_avg)))
fig1.show()

## Task 2 — Bottom vs. Top: A Contrast Story

In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
top_mean = top8['Happiness_Score'].mean()
bottom_mean = bottom8['Happiness_Score'].mean()
ratio = top_mean / bottom_mean

colors = [HIGHLIGHT if g == 'Top 8' else WARM for g in combined['Group']]

fig2 = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.2f}' for v in combined['Happiness_Score']],
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}<br>Score: %{x:.2f}<extra></extra>',
))

xmax = combined['Happiness_Score'].max() * 1.15

# Stretch goal: global average reference line
fig2.add_vline(x=global_avg, line_dash='dot', line_color='#475569',
               annotation_text=f'Global avg {global_avg:.2f}',
               annotation_position='top',
               annotation_font=dict(size=12, color='#475569'))

# Visual separator between the two groups (8 bottom bars sit at positions 0-7)
fig2.add_hline(y=7.5, line_dash='dash', line_color='#9CA3AF', line_width=1.5)

# Annotation that emphasises the gap
fig2.add_annotation(
    x=xmax * 0.98, y=7.5, xanchor='right', yanchor='middle',
    text=(f'<b>Gap: {top_mean - bottom_mean:.1f} points</b><br>'
          f'Top 8 average is {ratio:.1f}x the bottom 8'),
    showarrow=False, align='right', bgcolor='white',
    bordercolor='#9CA3AF', borderwidth=1, borderpad=6, font=dict(size=13),
)

fig2.update_layout(
    title=dict(text=(f"<b>The happiest countries score about {ratio:.1f}x the least happy: "
                     f"{top_mean:.1f} vs {bottom_mean:.1f}</b>"
                     "<br><sup>Top 8 (teal) and bottom 8 (coral) countries by happiness score, "
                     "World Happiness Report 2023</sup>")),
)
fig2.update_xaxes(range=[0, xmax], title='Happiness score')  # zero baseline
style(fig2, height=640)
fig2.show()

## Stretch goal — Grouped bar chart: GDP vs. Freedom by region

Note: the dataset column is `GDP` (renamed in the setup cell), not `GDP_per_capita`.

In [ ]:
# Stretch goal — grouped bar chart
wanted = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']
available = set(df['Region'].unique())
regions = [r for r in wanted if r in available]

# Region labels vary between dataset versions; fall back to the 5 largest regions
if len(regions) < 5:
    print('Some requested regions not found, using the 5 largest regions instead.')
    print('Missing:', [r for r in wanted if r not in available])
    regions = df['Region'].value_counts().head(5).index.tolist()

factors = df[df['Region'].isin(regions)].groupby('Region')[['GDP', 'Freedom']].mean()
factors = factors.sort_values('GDP')   # ascending: highest GDP ends up at the top

# Insight: does the wealthiest region also lead on freedom?
gdp_leader = factors['GDP'].idxmax()
freedom_rank = int(factors['Freedom'].rank(ascending=False)[gdp_leader])
if freedom_rank == 1:
    headline = f'{gdp_leader} leads on both GDP and freedom'
else:
    headline = (f'{gdp_leader} leads on GDP but ranks #{freedom_rank} of {len(factors)} on freedom: '
                'wealth does not guarantee it')

fig3 = go.Figure()
fig3.add_bar(y=factors.index, x=factors['GDP'], name='GDP (economic output)',
             orientation='h', marker_color='#2563EB',
             text=[f'{v:.2f}' for v in factors['GDP']], textposition='outside', cliponaxis=False)
fig3.add_bar(y=factors.index, x=factors['Freedom'], name='Freedom (life choices)',
             orientation='h', marker_color='#F59E0B',
             text=[f'{v:.2f}' for v in factors['Freedom']], textposition='outside', cliponaxis=False)

fig3.update_layout(
    barmode='group',
    title=dict(text=f'<b>{headline}</b><br><sup>Average GDP and Freedom sub-scores by region, '
                    'World Happiness Report 2023</sup>'),
)
style(fig3, height=560)
fig3.update_layout(showlegend=True,
                   legend=dict(orientation='h', yanchor='bottom', y=1.0, x=0, title_text=''))
fig3.update_xaxes(range=[0, factors.max().max() * 1.2], title='Average sub-score')  # zero baseline
fig3.show()